# 布林带均值回归策略 (Bollinger Bands Mean Reversion)

## 策略描述

布林带由 John Bollinger 在 1980 年代提出，是一种基于**标准差**的动态通道指标。
核心思想：价格在统计上倾向于回归均值，当价格触及通道边缘时反向交易。

## 计算公式

**中轨（Middle Band）**

$$ \text{MID}_t = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i} $$

其中 $n = 20$（默认周期），$P_t$ 为第 $t$ 日收盘价。

**标准差（Standard Deviation）**

$$ \sigma_t = \sqrt{\frac{1}{n} \sum_{i=0}^{n-1} (P_{t-i} - \text{MID}_t)^2} $$

**上轨和下轨（Upper / Lower Band）**

$$ \text{UPPER}_t = \text{MID}_t + k \cdot \sigma_t $$
$$ \text{LOWER}_t = \text{MID}_t - k \cdot \sigma_t $$

其中 $k = 2$（默认倍数）。正态分布下，价格在通道内的概率约为 95%。

**交易信号**

$$ \text{Signal}(t) = \begin{cases}
\text{Buy}  & \text{if } P_t \leq \text{LOWER}_t \text{ and not in position} \\
\text{Sell} & \text{if } P_t \geq \text{MID}_t \text{ and in position} \\
\text{Hold} & \text{otherwise}
\end{cases} $$

- 买入：价格跌破下轨 → 超卖，预期反弹
- 卖出：价格回到中轨 → 均值回归到位，止盈

## 参数

| 参数 | 值 | 说明 |
|------|----|------|
| 窗口周期 | 20 | 布林带计算窗口（交易日） |
| 标准差倍数 | 2 | 通道宽度 |
| 标的 | 000001 | 平安银行 |
| 数据区间 | 2015-01 ~ 2026-05 | 避免前复权早期低价失真 |
| 本金 | ¥100,000 | 初始资金 |
| 复权 | 前复权 (qfq) | 调整分红影响 |

In [ ]:
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
open System
open System.IO
open System.Diagnostics
open Plotly.NET

In [ ]:
// 辅助函数：运行外部进程并获取标准输出
let run cmd args =
    let p = new Process()
    p.StartInfo.FileName <- cmd
    p.StartInfo.Arguments <- args
    p.StartInfo.RedirectStandardOutput <- true
    p.StartInfo.UseShellExecute <- false
    let o = p.StandardOutput.ReadToEnd()
    p.WaitForExit()
    o

// 策略参数
let sym = "000001"

// 数据目录：notebooks/data/
let cwd = Environment.CurrentDirectory
Directory.CreateDirectory(Path.Combine(cwd, "data")) |> ignore
let csv = Path.Combine(cwd, "data", sym + ".csv")
let scr = Path.GetFullPath(Path.Combine(cwd, "..", "scripts", "fetch", "akshare_daily.py"))

// 如果 CSV 不存在，通过 akshare 下载
if not (File.Exists csv) then
    printfn "正在下载 %s 历史行情..." sym
    printf "%s" (run "python3" (scr + " " + sym))

In [ ]:
// K 线数据结构
type Bar = { D: DateTime; O: decimal; H: decimal; L: decimal; C: decimal; V: decimal }

// 加载 CSV 文件
// akshare 列顺序：日期,股票代码,开盘,收盘,最高,最低,成交量
let load path =
    let lines = File.ReadAllLines path
    lines.[1..] |> Array.map (fun line ->
        let p = line.Split(',')
        { D = DateTime.Parse(p.[0]); O = decimal p.[2]
          C = decimal p.[3]; H = decimal p.[4]
          L = decimal p.[5]; V = decimal p.[6] })

// 加载数据，过滤前复权早期产生的负价格
// 从 2015 年开始，避免前复权导致 1990 年代股价过低（<1 元）使头寸失真
let startDate = DateTime(2015, 1, 1)
let bars : Bar[] = load csv |> Array.filter (fun b -> b.C > 0m && b.D >= startDate)
printfn "已加载 %d 条记录" bars.Length
printfn "  时间范围: %s ~ %s" (bars.[0].D.ToString("yyyy-MM-dd")) ((Array.last bars).D.ToString("yyyy-MM-dd"))

In [ ]:
// 计算样本标准差
let stddev (data: decimal[]) =
    let n = float data.Length
    let avg = (data |> Array.sum) / decimal n
    let sq = data |> Array.sumBy (fun x -> (x - avg) * (x - avg))
    System.Math.Sqrt(float sq / n)

// 交易信号类型
type Action = Buy | Sell | Hold

// 布林带均值回归回测
let backtest () =
    let mutable cash = 100000m  // 初始现金
    let mutable pos = 0m       // 持仓股数
    let mutable buf : decimal[] = [||]  // 滚动窗口缓冲区
    let equity = ResizeArray()  // 权益曲线
    let trades = ResizeArray()  // 交易记录

    for b in bars do
        // 维护 20 日滚动窗口
        if buf.Length < 20 then
            buf <- Array.append buf [|b.C|]
        else
            buf <- Array.append buf.[1..] [|b.C|]

            // 计算布林带
            let avg = (buf |> Array.sum) / 20m          // 中轨 = SMA(20)
            let sd = decimal (stddev buf)               // 标准差
            let upper = avg + 2m * sd                   // 上轨 = MID + 2σ
            let lower = avg - 2m * sd                   // 下轨 = MID - 2σ

            // 交易规则：
            //   价格跌破下轨 → 买入（超卖反弹）
            //   价格回到中轨 → 卖出（均值回归到位）
            if b.C <= lower && cash > 0m then
                pos <- cash / b.C
                cash <- 0m
                trades.Add("BUY  " + b.D.ToString("yyyy-MM-dd") + " @ " + b.C.ToString("F2"))
            elif b.C >= avg && pos > 0m then
                cash <- pos * b.C
                pos <- 0m
                trades.Add("SELL " + b.D.ToString("yyyy-MM-dd") + " @ " + b.C.ToString("F2"))

        // 记录每日权益
        equity.Add(b.D, float (cash + pos * b.C))

    // 计算最终收益
    let lastC = (Array.last bars).C
    let final = cash + pos * lastC
    let ret = (final - 100000m) / 100000m * 100m
    ret, equity, trades

In [ ]:
// 执行回测
let ret, equity, trades = backtest ()

// 输出结果
printfn "========================================"
printfn "  布林带均值回归回测结果"
printfn "  标的: %s (平安银行)" sym
printfn "  总收益率: %.2f%%" ret
printfn "  交易次数: %d" trades.Count
printfn ""
printfn "  最近 10 笔交易:"
let start = max 0 (trades.Count - 10)
for i = start to trades.Count - 1 do
    printfn "    %s" trades.[i]
printfn "========================================"

In [ ]:
// 绘制权益曲线
Chart.Line(
    equity |> Seq.map (fun (d, _) -> d),
    equity |> Seq.map (fun (_, e) -> e))
|> Chart.withTitle "Bollinger Bands Mean Reversion - Equity Curve"
|> Chart.withXAxisStyle "Date"
|> Chart.withYAxisStyle "Equity (CNY)"